# Phase 1: SQL Analytics Layer (Business Intelligence Foundation)
This notebook implements the relational database design, loads the raw clinical and financial data into MySQL, enforces data integrity constraints, creates indexes for query optimization, and performs a series of business-intelligence queries requested by hospital leadership.

### Import Dependencies and Load Environment Settings

In [1]:
import os
import pandas as pd
import urllib.parse
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Load connection settings from .env file
load_dotenv(dotenv_path="../.env")

host = os.getenv("MYSQL_HOST", "localhost")
port = os.getenv("MYSQL_PORT", "3306")
user = os.getenv("MYSQL_USER", "root")
password = os.getenv("MYSQL_PASSWORD", "")
db = os.getenv("MYSQL_DATABASE", "hospital_db")

# Create connection engine
password_encoded = urllib.parse.quote_plus(password)
conn_str = f"mysql+pymysql://{user}:{password_encoded}@{host}:{port}/{db}"
engine = create_engine(conn_str)
print(f"Connected successfully to MySQL database '{db}' on {host}:{port}!")

Connected successfully to MySQL database 'hospital_db' on localhost:3306!


### Relational Schema Design and Table Creation
To build a robust foundation, we define our primary keys, foreign keys, and relational constraints (e.g. `ON DELETE CASCADE`) at the database level. This ensures data integrity and consistency.

In [2]:
with engine.connect() as conn:
    # Drop existing tables in reverse order of foreign keys
    conn.execute(text("DROP TABLE IF EXISTS billing;"))
    conn.execute(text("DROP TABLE IF EXISTS visits;"))
    conn.execute(text("DROP TABLE IF EXISTS patients;"))
    conn.commit()
    print("Dropped pre-existing tables to reset database schema.")

create_patients = """
CREATE TABLE patients (
    patient_id INT PRIMARY KEY,
    age INT,
    gender VARCHAR(10),
    city VARCHAR(50),
    insurance_provider VARCHAR(100),
    chronic_flag TINYINT,
    registration_date DATE
);
"""

create_visits = """
CREATE TABLE visits (
    visit_id INT PRIMARY KEY,
    patient_id INT,
    visit_date DATE,
    department VARCHAR(50),
    visit_type VARCHAR(20),
    length_of_stay_hours DECIMAL(10, 2),
    risk_score VARCHAR(20),
    doctor_id INT,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id) ON DELETE CASCADE
);
"""

create_billing = """
CREATE TABLE billing (
    bill_id INT PRIMARY KEY,
    visit_id INT,
    billed_amount DECIMAL(10, 2),
    approved_amount DECIMAL(10, 2),
    claim_status VARCHAR(20),
    payment_days INT,
    billing_date DATE,
    FOREIGN KEY (visit_id) REFERENCES visits(visit_id) ON DELETE CASCADE
);
"""

with engine.connect() as conn:
    conn.execute(text(create_patients))
    conn.execute(text(create_visits))
    conn.execute(text(create_billing))
    conn.commit()
    print("Tables created successfully with primary keys and foreign key constraints!")

Dropped pre-existing tables to reset database schema.


Tables created successfully with primary keys and foreign key constraints!


### Data Cleaning and Ingestion
We read the source CSV files into memory. Blank or empty cells in numerical columns (`approved_amount`, `payment_days`, `length_of_stay_hours`) are cleaned and loaded as SQL `NULL` instead of empty strings or zeros, ensuring we do not distort clinical or financial metrics.

In [3]:
# Read data
patients_df = pd.read_csv("../data/patients.csv")
visits_df = pd.read_csv("../data/visits.csv")
billing_df = pd.read_csv("../data/billing.csv")

# Clean missing values (convert NaN/empty to Python None, which maps to SQL NULL)
patients_df = patients_df.where(pd.notnull(patients_df), None)

visits_df['length_of_stay_hours'] = pd.to_numeric(visits_df['length_of_stay_hours'], errors='coerce')
visits_df = visits_df.where(pd.notnull(visits_df), None)

billing_df['billed_amount'] = pd.to_numeric(billing_df['billed_amount'], errors='coerce')
billing_df['approved_amount'] = pd.to_numeric(billing_df['approved_amount'], errors='coerce')
billing_df['payment_days'] = pd.to_numeric(billing_df['payment_days'], errors='coerce')
billing_df = billing_df.where(pd.notnull(billing_df), None)

print(f"Loaded raw rows: Patients={len(patients_df)}, Visits={len(visits_df)}, Billing={len(billing_df)}")

# Ingest in relational order
print("Ingesting patients table...")
patients_df.to_sql('patients', con=engine, if_exists='append', index=False)

print("Ingesting visits table...")
visits_df.to_sql('visits', con=engine, if_exists='append', index=False)

print("Ingesting billing table...")
billing_df.to_sql('billing', con=engine, if_exists='append', index=False)

print("Data ingestion complete!")

Loaded raw rows: Patients=5000, Visits=25000, Billing=25000
Ingesting patients table...


Ingesting visits table...


Ingesting billing table...


Data ingestion complete!


### Indexing Strategy
Creating database indexes speeds up data retrieval. We create indexes on filtering and grouping columns: `visit_date`, `department`, `insurance_provider`, and `claim_status`.

In [4]:
with engine.connect() as conn:
    conn.execute(text("CREATE INDEX idx_visit_date ON visits(visit_date);"))
    conn.execute(text("CREATE INDEX idx_department ON visits(department);"))
    # For VARCHAR index in MySQL, specifying prefix length is recommended if long, otherwise standard index is fine.
    conn.execute(text("CREATE INDEX idx_insurance_provider ON patients(insurance_provider(50));"))
    conn.execute(text("CREATE INDEX idx_claim_status ON billing(claim_status);"))
    conn.commit()
    print("Database indexes created successfully!")

Database indexes created successfully!


### SQL Business Queries
We execute the 15 required business intelligence queries requested by leadership.

#### Operational Analysis (Queries 1-5)

In [5]:
print("--- 1. Top 10 Departments by Visit Volume ---")
q1 = """
SELECT department, COUNT(*) as visit_volume
FROM visits
GROUP BY department
ORDER BY visit_volume DESC
LIMIT 10;
"""
print(pd.read_sql(q1, engine))

--- 1. Top 10 Departments by Visit Volume ---


    department  visit_volume
0      General          4228
1           ER          4220
2    Neurology          4165
3  Orthopedics          4164
4   Cardiology          4159
5          ICU          4064


In [6]:
print("\n--- 2. Top 5 Departments by Average Length of Stay ---")
q2 = """
SELECT department, ROUND(AVG(length_of_stay_hours), 2) as avg_length_of_stay_hours
FROM visits
GROUP BY department
ORDER BY avg_length_of_stay_hours DESC
LIMIT 5;
"""
print(pd.read_sql(q2, engine))


--- 2. Top 5 Departments by Average Length of Stay ---
    department  avg_length_of_stay_hours
0    Neurology                     19.72
1  Orthopedics                     19.66
2   Cardiology                     19.60
3           ER                     19.53
4      General                     19.43


In [7]:
print("\n--- 3. Percentage of High Risk Visits per Department ---")
q3 = """
SELECT department,
       COUNT(*) as total_visits,
       SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) as high_risk_visits,
       ROUND((SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)), 2) as high_risk_percentage
FROM visits
GROUP BY department
ORDER BY high_risk_percentage DESC;
"""
print(pd.read_sql(q3, engine))


--- 3. Percentage of High Risk Visits per Department ---


    department  total_visits  high_risk_visits  high_risk_percentage
0          ICU          4064             845.0                 20.79
1           ER          4220             872.0                 20.66
2    Neurology          4165             846.0                 20.31
3  Orthopedics          4164             842.0                 20.22
4      General          4228             839.0                 19.84
5   Cardiology          4159             790.0                 18.99


In [8]:
print("\n--- 4. Average Number of Visits per Patient by City ---")
q4 = """
SELECT p.city,
       COUNT(v.visit_id) as total_visits,
       COUNT(DISTINCT p.patient_id) as unique_patients,
       ROUND((COUNT(v.visit_id) * 1.0 / COUNT(DISTINCT p.patient_id)), 2) as avg_visits_per_patient
FROM patients p
LEFT JOIN visits v ON p.patient_id = v.patient_id
GROUP BY p.city
ORDER BY avg_visits_per_patient DESC;
"""
print(pd.read_sql(q4, engine))


--- 4. Average Number of Visits per Patient by City ---


        city  total_visits  unique_patients  avg_visits_per_patient
0       Pune          4221              831                    5.08
1  Hyderabad          4370              869                    5.03
2  Bangalore          4205              840                    5.01
3     Mumbai          4122              822                    5.01
4    Chennai          3975              801                    4.96
5      Delhi          4107              837                    4.91


In [9]:
print("\n--- 5. Doctors Handling the Most High Risk Visits ---")
q5 = """
SELECT doctor_id, COUNT(*) as high_risk_visits_handled
FROM visits
WHERE risk_score = 'High'
GROUP BY doctor_id
ORDER BY high_risk_visits_handled DESC
LIMIT 10;
"""
print(pd.read_sql(q5, engine))


--- 5. Doctors Handling the Most High Risk Visits ---


   doctor_id  high_risk_visits_handled
0        174                        71
1        198                        69
2        169                        68
3        177                        67
4        135                        65
5        105                        65
6        188                        64
7        180                        64
8        131                        62
9        178                        61


#### Financial Analysis (Queries 6-10)

In [10]:
print("--- 6. Top 10 Insurance Providers by Total Billed Amount ---")
q6 = """
SELECT p.insurance_provider, ROUND(SUM(b.billed_amount), 2) as total_billed_amount
FROM patients p
JOIN visits v ON p.patient_id = v.patient_id
JOIN billing b ON v.visit_id = b.visit_id
GROUP BY p.insurance_provider
ORDER BY total_billed_amount DESC
LIMIT 10;
"""
print(pd.read_sql(q6, engine))

--- 6. Top 10 Insurance Providers by Total Billed Amount ---


  insurance_provider  total_billed_amount
0          MediCareX         1.345912e+08
1            CareOne         1.307080e+08
2         HealthPlus         1.301807e+08
3         SecureLife         1.262890e+08


In [11]:
print("\n--- 7. Top 5 Insurance Providers with Highest Claim Rejection Rates ---")
q7 = """
SELECT p.insurance_provider,
       COUNT(b.bill_id) as total_claims,
       SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END) as rejected_claims,
       ROUND((SUM(CASE WHEN b.claim_status = 'Rejected' THEN 1 ELSE 0 END) * 100.0 / COUNT(b.bill_id)), 2) as rejection_rate
FROM patients p
JOIN visits v ON p.patient_id = v.patient_id
JOIN billing b ON v.visit_id = b.visit_id
GROUP BY p.insurance_provider
ORDER BY rejection_rate DESC
LIMIT 5;
"""
print(pd.read_sql(q7, engine))


--- 7. Top 5 Insurance Providers with Highest Claim Rejection Rates ---


  insurance_provider  total_claims  rejected_claims  rejection_rate
0         SecureLife          5965            936.0           15.69
1          MediCareX          6532            996.0           15.25
2         HealthPlus          6220            931.0           14.97
3            CareOne          6283            934.0           14.87


In [12]:
print("\n--- 8. Average Payment Delay by Insurance Provider ---")
q8 = """
SELECT p.insurance_provider,
       ROUND(AVG(b.payment_days), 1) as avg_payment_delay_days
FROM patients p
JOIN visits v ON p.patient_id = v.patient_id
JOIN billing b ON v.visit_id = b.visit_id
WHERE b.payment_days IS NOT NULL
GROUP BY p.insurance_provider
ORDER BY avg_payment_delay_days DESC;
"""
print(pd.read_sql(q8, engine))


--- 8. Average Payment Delay by Insurance Provider ---


  insurance_provider  avg_payment_delay_days
0         SecureLife                    13.1
1         HealthPlus                    13.1
2            CareOne                    13.0
3          MediCareX                    13.0


In [13]:
print("\n--- 9. Revenue Realization Ratio by Department ---")
q9 = """
SELECT v.department,
       ROUND(SUM(b.billed_amount), 2) as total_billed,
       ROUND(SUM(b.approved_amount), 2) as total_approved,
       ROUND((SUM(b.approved_amount) / NULLIF(SUM(b.billed_amount), 0)), 4) as revenue_realization_ratio
FROM visits v
JOIN billing b ON v.visit_id = b.visit_id
GROUP BY v.department
ORDER BY revenue_realization_ratio DESC;
"""
print(pd.read_sql(q9, engine))


--- 9. Revenue Realization Ratio by Department ---


    department  total_billed  total_approved  revenue_realization_ratio
0          ICU   84757763.76     63166516.84                     0.7453
1  Orthopedics   87811455.80     65211585.83                     0.7426
2      General   87131451.86     64690870.95                     0.7425
3    Neurology   87310048.09     64708778.69                     0.7411
4           ER   88686960.35     65672329.38                     0.7405
5   Cardiology   86071256.19     63705806.68                     0.7402

In [14]:
print("\n--- 10. High Billed Amount (> Average) with Zero or Missing Approval ---")
q10 = """
SELECT v.visit_id, v.visit_date, v.department, ROUND(b.billed_amount, 2) as billed_amount, ROUND(b.approved_amount, 2) as approved_amount, b.claim_status
FROM visits v
JOIN billing b ON v.visit_id = b.visit_id
WHERE b.billed_amount > (SELECT AVG(billed_amount) FROM billing)
  AND (b.approved_amount = 0 OR b.approved_amount IS NULL)
ORDER BY b.billed_amount DESC
LIMIT 10;
"""
print(pd.read_sql(q10, engine))


--- 10. High Billed Amount (> Average) with Zero or Missing Approval ---
   visit_id  visit_date   department  billed_amount  approved_amount  \
0      1570  2025-07-23      General       78054.79              NaN   
1     18381  2025-12-29      General       68213.53              0.0   
2     15092  2025-11-25    Neurology       66410.33              NaN   
3      2638  2025-07-03    Neurology       65167.44              NaN   
4      8089  2025-04-08           ER       62661.81              NaN   
5     19310  2025-12-07          ICU       60510.22              NaN   
6     24623  2025-05-17    Neurology       58275.40              NaN   
7     18189  2025-07-29    Neurology       57521.12              NaN   
8     14702  2025-12-06  Orthopedics       56848.91              NaN   
9      5131  2026-01-02  Orthopedics       56736.52              NaN   

  claim_status  
0         Paid  
1     Rejected  
2         Paid  
3         Paid  
4         Paid  
5         Paid  
6         Paid

#### Data Quality & Integrity Audits (Queries 11-15)

In [15]:
print("--- 11. Visits Without Billing Records (Orphans) ---")
q11 = """
SELECT v.visit_id, v.visit_date, v.patient_id, v.department
FROM visits v
LEFT JOIN billing b ON v.visit_id = b.visit_id
WHERE b.bill_id IS NULL;
"""
df_o1 = pd.read_sql(q11, engine)
print(f"Orphan visits: {len(df_o1)}")
print(df_o1.head(10))

--- 11. Visits Without Billing Records (Orphans) ---
Orphan visits: 0
Empty DataFrame
Columns: [visit_id, visit_date, patient_id, department]
Index: []


In [16]:
print("\n--- 12. Billing Records Without Corresponding Visits ---")
q12 = """
SELECT b.bill_id, b.visit_id, b.billed_amount, b.billing_date
FROM billing b
LEFT JOIN visits v ON b.visit_id = v.visit_id
WHERE v.visit_id IS NULL;
"""
df_o2 = pd.read_sql(q12, engine)
print(f"Orphan bills: {len(df_o2)}")
print(df_o2.head(10))


--- 12. Billing Records Without Corresponding Visits ---
Orphan bills: 0
Empty DataFrame
Columns: [bill_id, visit_id, billed_amount, billing_date]
Index: []


In [17]:
print("\n--- 13. Duplicate Patient Records in Patients Table ---")
q13 = """
SELECT patient_id, COUNT(*) as duplicate_count
FROM patients
GROUP BY patient_id
HAVING duplicate_count > 1;
"""
df_dup = pd.read_sql(q13, engine)
print(f"Duplicate patients: {len(df_dup)}")
print(df_dup)


--- 13. Duplicate Patient Records in Patients Table ---
Duplicate patients: 0
Empty DataFrame
Columns: [patient_id, duplicate_count]
Index: []


In [18]:
print("\n--- 14. Invalid Length of Stay or Payment Days ---")
q14a = """
SELECT visit_id, length_of_stay_hours
FROM visits
WHERE length_of_stay_hours IS NULL OR length_of_stay_hours <= 0;
"""
df_inv_los = pd.read_sql(q14a, engine)
print(f"Invalid length of stay records: {len(df_inv_los)}")

q14b = """
SELECT bill_id, claim_status, payment_days
FROM billing
WHERE payment_days < 0 OR (claim_status = 'Paid' AND payment_days IS NULL);
"""
df_inv_pd = pd.read_sql(q14b, engine)
print(f"Invalid payment days records: {len(df_inv_pd)}")


--- 14. Invalid Length of Stay or Payment Days ---
Invalid length of stay records: 0


Invalid payment days records: 459


In [19]:
print("\n--- 15. Visits with Patients Missing Insurance Information ---")
q15 = """
SELECT v.visit_id, v.patient_id, p.insurance_provider
FROM visits v
JOIN patients p ON v.patient_id = p.patient_id
WHERE p.insurance_provider IS NULL OR TRIM(p.insurance_provider) = '' OR LOWER(p.insurance_provider) = 'null';
"""
df_miss_ins = pd.read_sql(q15, engine)
print(f"Visits with missing insurance: {len(df_miss_ins)}")
print(df_miss_ins.head(10))


--- 15. Visits with Patients Missing Insurance Information ---


Visits with missing insurance: 0
Empty DataFrame
Columns: [visit_id, patient_id, insurance_provider]
Index: []
